In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import mlflow
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt


In [2]:
df = pd.read_csv('/Users/andresrodartee/mlops-and-system-design/session_2/datasets2/Churn_Modelling_val.csv')
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,8607,15694581,Rawlings,807,Spain,Male,42.0,5,0.00,2,1.0,1.0,74900.90,0
1,4685,15736963,Herring,623,France,Male,43.0,1,0.00,2,1.0,1.0,146379.30,0
2,1732,15721730,Amechi,601,Spain,Female,44.0,4,0.00,2,1.0,0.0,58561.31,0
3,4743,15762134,Liang,506,Germany,Male,59.0,8,119152.10,2,1.0,1.0,170679.74,0
4,4522,15648898,Chuang,560,Spain,Female,27.0,7,124995.98,1,1.0,1.0,114669.79,0


In [3]:
def balance_dataset(df: pd.DataFrame) -> pd.DataFrame:

    df_y0 = df[df['Exited'] == 0]
    df_y1 = df[df['Exited'] == 1]

    min_size = len(df_y1)
    df_y0_balanced = df_y0.sample(n=min_size, random_state=42)

    df_balanced = pd.concat([df_y0_balanced, df_y1], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)

    return df_balanced

df = balance_dataset(df)

In [4]:
class Transformer:
    def __init__(self):
        self.DROP_COLUMNS = [
            'CustomerId',
            'RowNumber',
            'Surname'
        ]
        self.BINARY_FEATURES = [
           "Gender"
        ]
    
    def _binary_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        for feature in self.BINARY_FEATURES:
            df[feature] = df[feature].map({'Male': 0, 'Female': 1})
        return df
    
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.drop(self.DROP_COLUMNS, axis=1)
        df = self._binary_transform(df)
        return df

In [5]:
df = Transformer().transform(df)
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,816,Germany,1,34.0,2,108410.87,2,1.0,0.0,102908.91,0
1,723,Germany,1,47.0,10,90450.00,2,0.0,0.0,103379.31,1
2,706,Spain,1,53.0,3,0.00,3,0.0,0.0,88479.02,1
3,679,Germany,0,52.0,9,135870.01,2,0.0,0.0,54038.62,0
4,632,Germany,0,41.0,3,126550.70,1,0.0,0.0,177644.52,1


In [6]:
one_hot_encode_columns = [
    'Geography'
]

encoder = OneHotEncoder(sparse_output=False, drop='first').set_output(transform='pandas')
encoder.fit(df[one_hot_encode_columns])
encoded_df = encoder.transform(df[one_hot_encode_columns])

df = df.drop(columns=one_hot_encode_columns)
df = pd.concat([df, encoded_df], axis=1)
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_Germany,Geography_Spain
0,816,1,34.0,2,108410.87,2,1.0,0.0,102908.91,0,1.0,0.0
1,723,1,47.0,10,90450.00,2,0.0,0.0,103379.31,1,1.0,0.0
2,706,1,53.0,3,0.00,3,0.0,0.0,88479.02,1,0.0,1.0
3,679,0,52.0,9,135870.01,2,0.0,0.0,54038.62,0,1.0,0.0
4,632,0,41.0,3,126550.70,1,0.0,0.0,177644.52,1,1.0,0.0


In [7]:
y_val = df['Exited']
X_val = df.drop('Exited', axis=1)

In [8]:
def batch_inference(model_uri, X_val, y_val):
    print(f"Loading model from: {model_uri}")
    
    model = mlflow.pyfunc.load_model(model_uri)

In [11]:
def batch_inference(model_uri, X_val, y_val):
    print(f"Loading model from: {model_uri}")
    
    model = mlflow.pyfunc.load_model(model_uri)
    
    predictions = model.predict(X_val)
    
    cm = confusion_matrix(y_val, predictions)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    
    fig, ax = plt.subplots(figsize=(6, 6))
    disp.plot(cmap=plt.cm.Blues, ax=ax)
    plt.title("Batch Inference Confusion Matrix")
    plt.show()
    
    return predictions

In [13]:
my_uri = 'runs:/cd82ad44aca54d0bbc24f96d86614f07/churn_model'
predictions = batch_inference(my_uri, X_val, y_val)

Loading model from: runs:/cd82ad44aca54d0bbc24f96d86614f07/churn_model


MlflowException: Run with id=cd82ad44aca54d0bbc24f96d86614f07 not found